In [ ]:
import pandas as pd
import sqlite3 as sql

# ---------- Merging and Joining ----------*
# 1
with sql.connect('chinook.db') as connection:
  df_customers = pd.read_sql_query('SELECT * FROM customers', con=connection)
  df_invoices = pd.read_sql_query('SELECT * FROM invoices', con=connection)
  result = pd.merge(df_customers, df_invoices, on='CustomerId')
  total_invoices_per_customer = result[['FirstName', 'Total']]
  print(total_invoices_per_customer)
  total_invoices = result['Total'].sum()
  print(total_invoices)


    FirstName  Total
0        Luís   3.98
1        Luís   3.96
2        Luís   5.94
3        Luís   0.99
4        Luís   1.98
..        ...    ...
407      Puja   5.94
408      Puja   1.99
409      Puja   1.98
410      Puja  13.86
411      Puja   8.91

[412 rows x 2 columns]
2328.6


In [ ]:
# ---------- Merging and Joining ----------*
# 2
df_movie = pd.read_csv('movie.csv')
df_color = df_movie[['director_name', 'color']].dropna()
display(df_color.head())
df_reviews = df_movie[['director_name', 'num_critic_for_reviews']].dropna()
display(df_reviews.head())
left_result = pd.merge(df_color, df_reviews, on='director_name', how='left').reset_index()
full_outer_result = pd.merge(df_color, df_reviews, on='director_name', how='outer').reset_index()
display(left_result.sample(10))
print(left_result.shape)
display(full_outer_result.sample(10))
print(full_outer_result.shape)

,director_name,color
0,James Cameron,Color
1,Gore Verbinski,Color
2,Sam Mendes,Color
3,Christopher Nolan,Color
5,Andrew Stanton,Color


,director_name,num_critic_for_reviews
0,James Cameron,723.0
1,Gore Verbinski,302.0
2,Sam Mendes,602.0
3,Christopher Nolan,813.0
5,Andrew Stanton,462.0


,index,director_name,color,num_critic_for_reviews
1139,1139,Peyton Reed,Color,183.0
11404,11404,Clint Eastwood,Color,268.0
11954,11954,Todd Phillips,Color,334.0
14140,14140,James Ivory,Color,58.0
8590,8590,Simon West,Color,152.0
14975,14975,Reginald Hudlin,Color,64.0
10260,10260,Barry Levinson,Color,111.0
8293,8293,Alexandre Aja,Color,251.0
18549,18549,John Laing,Color,NaN
4011,4011,Steven Spielberg,Color,538.0


(19852, 4)


,index,director_name,color,num_critic_for_reviews
7290,7290,Joel Schumacher,Color,183.0
5633,5633,Gore Verbinski,Color,450.0
8406,8406,Jonathan Liebesman,Color,348.0
9702,9702,Marc Forster,Color,253.0
17790,17790,Taylor Hackford,Color,12.0
2345,2345,Chris Columbus,Color,217.0
12787,12787,Phillip Noyce,Color,47.0
18857,18857,Vincenzo Natali,Color,38.0
8974,8974,Kevin Smith,Black and White,254.0
18239,18239,Timur Bekmambetov,Color,393.0


(19860, 4)


In [ ]:
#  --------Grouping and Aggregating---------
# 1
df_excel = pd.read_excel('titanic.xlsx')
display(df_excel.head())
result = df_excel.groupby('Pclass').agg({
    'Age': ['mean'],
    'Fare': ['sum'],
}).reset_index()
result.columns = ['_'.join(col) for col in result.columns]
result = result.astype(int)
new_df = result
display(new_df.sample(10, replace="False"))

,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NaN,S
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C
2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,C123,S
4,5,0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,NaN,S


,Pclass_,Age_mean,Fare_sum
0,1,38,18177
0,1,38,18177
0,1,38,18177
1,2,29,3801
0,1,38,18177
1,2,29,3801
0,1,38,18177
0,1,38,18177
0,1,38,18177
0,1,38,18177


In [ ]:
# ---------- Grouping and Aggregating ---------
# 2
grouped_movie = df_movie.groupby(['color', 'director_name']).agg({
    'num_critic_for_reviews': 'sum',
    'duration': 'mean'
}).reset_index()


grouped_movie.columns = ['color', 'director_name', 'total_critic_reviews', 'average_duration']

display(grouped_movie.sample(10))

,color,director_name,total_critic_reviews,average_duration
2239,Color,Steve Carver,29.0,107.0
1483,Color,Lenny Abrahamson,421.0,118.0
369,Color,Billy Kent,50.0,88.0
393,Color,Brad Anderson,804.0,100.6
2206,Color,Stanley Kubrick,942.0,162.5
1809,Color,Oliver Blackburn,114.0,99.0
2412,Color,Vidhu Vinod Chopra,22.0,101.0
1722,Color,Mike Mitchell,535.0,92.0
780,Color,Elaine May,49.0,107.0
1158,Color,Jerry Zaks,45.0,98.0


In [ ]:
# ---------- Grouping and Aggregating ---------
# 3
df_parquit = pd.read_parquet('./drive/MyDrive/data/flights/')

# Convert columns to numeric, coercing errors to NaN
df_parquit['ArrDelay'] = pd.to_numeric(df_parquit['ArrDelay'], errors='coerce')
df_parquit['DepDelay'] = pd.to_numeric(df_parquit['DepDelay'], errors='coerce')

result = df_parquit.groupby(['Year', 'Month']).agg({
    "ArrDelay": 'mean',
    "DepDelay": "max",
}).reset_index()

# Perform sum only on numeric columns to avoid TypeErrors
result_total = df_parquit.groupby(['Year', 'Month']).sum(numeric_only=True).reset_index()

display(result)
display(result_total)

In [15]:
# --------Applying Functions-------
# 1
df_excel = pd.read_excel('titanic.xlsx')
display(df_excel.head())

def modify_age(row):
  age = "Child" if row['Age'] < 18 else "Adult"
  return age

df_excel['Age_group'] = df_excel.apply(modify_age, axis=1)
display(df_excel.sample(10))

,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NaN,S
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C
2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,C123,S
4,5,0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,NaN,S


,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked,Age_group
724,725,1,1,"Chambers, Mr. Norman Campbell",male,27.0,1,0,113806,53.1000,E8,S,Adult
338,339,1,3,"Dahl, Mr. Karl Edwart",male,45.0,0,0,7598,8.0500,NaN,S,Adult
115,116,0,3,"Pekoniemi, Mr. Edvard",male,21.0,0,0,STON/O 2. 3101294,7.9250,NaN,S,Adult
318,319,1,1,"Wick, Miss. Mary Natalie",female,31.0,0,2,36928,164.8667,C7,S,Adult
752,753,0,3,"Vande Velde, Mr. Johannes Joseph",male,33.0,0,0,345780,9.5000,NaN,S,Adult
888,889,0,3,"Johnston, Miss. Catherine Helen ""Carrie""",female,NaN,1,2,W./C. 6607,23.4500,NaN,S,Adult
315,316,1,3,"Nilsson, Miss. Helmina Josefina",female,26.0,0,0,347470,7.8542,NaN,S,Adult
617,618,0,3,"Lobb, Mrs. William Arthur (Cordelia K Stanlick)",female,26.0,1,0,A/5. 3336,16.1000,NaN,S,Adult
524,525,0,3,"Kassem, Mr. Fared",male,NaN,0,0,2700,7.2292,NaN,C,Adult
488,489,0,3,"Somerton, Mr. Francis William",male,30.0,0,0,A.5. 18509,8.0500,NaN,S,Adult


In [30]:
# --------Applying Functions-------
# 2
df_employee = pd.read_csv('employee.csv')
display(df_employee.head())

x_min = df_employee['BASE_SALARY'].min()
x_max = df_employee['BASE_SALARY'].max()

df_employee['normalized_salary'] = df_employee['BASE_SALARY'].apply(lambda x: (x - x_min) / (x_max - x_min))
display(df_employee.sample(10))

,UNIQUE_ID,POSITION_TITLE,DEPARTMENT,BASE_SALARY,RACE,EMPLOYMENT_TYPE,GENDER,EMPLOYMENT_STATUS,HIRE_DATE,JOB_DATE
0,0,ASSISTANT DIRECTOR (EX LVL),Municipal Courts Department,121862.0,Hispanic/Latino,Full Time,Female,Active,2006-06-12,2012-10-13
1,1,LIBRARY ASSISTANT,Library,26125.0,Hispanic/Latino,Full Time,Female,Active,2000-07-19,2010-09-18
2,2,POLICE OFFICER,Houston Police Department-HPD,45279.0,White,Full Time,Male,Active,2015-02-03,2015-02-03
3,3,ENGINEER/OPERATOR,Houston Fire Department (HFD),63166.0,White,Full Time,Male,Active,1982-02-08,1991-05-25
4,4,ELECTRICIAN,General Services Department,56347.0,White,Full Time,Male,Active,1989-06-19,1994-10-22


,UNIQUE_ID,POSITION_TITLE,DEPARTMENT,BASE_SALARY,RACE,EMPLOYMENT_TYPE,GENDER,EMPLOYMENT_STATUS,HIRE_DATE,JOB_DATE,normalized_salary
919,919,FIRE FIGHTER,Houston Fire Department (HFD),51194.0,White,Full Time,Male,Active,2009-07-27,2010-10-27,0.104919
615,615,SENIOR POLICE OFFICER,Houston Police Department-HPD,61643.0,Hispanic/Latino,Full Time,Male,Active,2002-01-28,2014-02-01,0.146709
1503,1503,ENGINEER/OPERATOR,Houston Fire Department (HFD),63166.0,White,Full Time,Male,Active,1997-11-03,2003-03-01,0.152800
1434,1434,ENGINEER/OPERATOR,Houston Fire Department (HFD),62540.0,Black or African American,Full Time,Male,Active,2004-11-08,2014-05-03,0.150296
1225,1225,SENIOR REFUSE TRUCK DRIVER,Solid Waste Management,32968.0,Black or African American,Full Time,Male,Active,2015-04-27,2015-04-27,0.032027
528,528,SENIOR POLICE OFFICER,Houston Police Department-HPD,66614.0,White,Full Time,Female,Active,1999-02-22,2011-09-03,0.166589
840,840,LIBRARY ASSISTANT,Library,26125.0,Black or African American,Full Time,Female,Active,2007-09-10,2007-09-10,0.004659
911,911,EQUIPMENT WORKER,Public Works & Engineering-PWE,32968.0,Black or African American,Full Time,Male,Active,2014-09-02,2014-09-02,0.032027
415,415,WEB DESIGNER,Solid Waste Management,55890.0,Hispanic/Latino,Full Time,Male,Active,2013-09-30,2013-09-30,0.123700
1886,1886,SENIOR SIDELOADER OPERATOR,Solid Waste Management,35630.0,Black or African American,Full Time,Male,Active,2014-11-24,2014-11-24,0.042673


In [31]:
import pandas as pd

# Load the movie dataset
df_movie = pd.read_csv('movie.csv')

# Define the classification function based on movie duration
def classify_duration(duration):
    if pd.isna(duration):
        return 'Unknown'
    elif duration < 60:
        return 'Short'
    elif duration <= 120:
        return 'Medium'
    else:
        return 'Long'

# Apply the custom function to create a new column
df_movie['duration_category'] = df_movie['duration'].apply(classify_duration)

# Display the original duration and the classification
display(df_movie[['movie_title', 'duration', 'duration_category']].head(10))

,movie_title,duration,duration_category
0,Avatar,178.0,Long
1,Pirates of the Caribbean: At World's End,169.0,Long
2,Spectre,148.0,Long
3,The Dark Knight Rises,164.0,Long
4,Star Wars: Episode VII - The Force Awakens,NaN,Unknown
5,John Carter,132.0,Long
6,Spider-Man 3,156.0,Long
7,Tangled,100.0,Medium
8,Avengers: Age of Ultron,141.0,Long
9,Harry Potter and the Half-Blood Prince,153.0,Long


In [55]:
# ---------Using pipe----------
import pandas as pd

df_titanic = pd.read_excel('titanic.xlsx')

# Fill missing Age values with the mean
df_titanic['Age'] = df_titanic['Age'].fillna(df_titanic['Age'].mean())

# Define functions for pipe
def filter_by_survival(df):
    return df[df['Survived'] == 1]

def add_fare_per_age(df, col_name):
    df[col_name] = df['Fare'] / df['Age']
    return df

# Chain operations using pipe
df_cleaned = (
    df_titanic
    .pipe(add_fare_per_age, 'Fare_Per_Age')
    .pipe(filter_by_survival)
)

display(df_cleaned.head())

,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked,Fare_Per_Age
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C,1.875876
2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S,0.304808
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,C123,S,1.517143
8,9,1,3,"Johnson, Mrs. Oscar W (Elisabeth Vilhelmina Berg)",female,27.0,0,2,347742,11.1333,NaN,S,0.412344
9,10,1,2,"Nasser, Mrs. Nicholas (Adele Achem)",female,14.0,1,0,237736,30.0708,NaN,C,2.147914
